In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 12 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 12.C1 — Check a partition contract

Given disjointness candidates and the full set of IDs, return whether train, validation, and test are pairwise disjoint and jointly exhaustive.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Set intersections test overlap; the union tests coverage.

</details>

In [3]:
def partition_valid(train,validation,test,all_ids):
    a,b,c=map(set,(train,validation,test))
    return not(a&b or a&c or b&c) and a|b|c==set(all_ids)

In [4]:
assert partition_valid([0,1],[2],[3],range(4))
assert not partition_valid([0,1],[1,2],[3],range(4))
splits=json.loads((ROOT/'data/splits.json').read_text())
assert partition_valid(splits['train'],splits['validation'],splits['test'],range(1797))
print("Frozen image-ID partitions are disjoint and exhaustive")

Frozen image-ID partitions are disjoint and exhaustive


**Why this works and what it does not establish:** ID disjointness does not guarantee independence. Images from the same writer or frames from one recording can still be related across partitions when those grouping variables are unavailable.

## 12.C2 — Report counts with accuracy

Return correct count, total count, and accuracy for nonempty equally sized true/predicted arrays.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Count equality element by element and divide by the number of observations.

</details>

In [5]:
def accuracy_counts(truth,prediction):
    truth=np.asarray(truth);prediction=np.asarray(prediction)
    correct=int(np.sum(truth==prediction));total=len(truth)
    return correct,total,correct/total

In [6]:
assert accuracy_counts([0,1,2],[0,0,2])==(2,3,2/3)
frame=pd.read_csv(ROOT/'reports/stage_12/test_predictions.csv')
correct,total,accuracy=accuracy_counts(frame.true_label,frame.pixels)
assert total==360
print("Previously exposed worked-test pixel result:",correct,total,accuracy)

Previously exposed worked-test pixel result: 347 360 0.9638888888888889


**Why this works and what it does not establish:** Counts make small apparent improvements easier to interpret. These are already exposed teaching-test predictions; inspecting them is not a newly blinded experiment.

## 12.C3 — Compare errors on the same observations

Return counts of observations where A alone is correct and where B alone is correct, in that order.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Use the same sample ordering for both models before applying boolean masks.

</details>

In [7]:
def paired_disagreements(truth,pred_a,pred_b):
    truth=np.asarray(truth);a=np.asarray(pred_a)==truth;b=np.asarray(pred_b)==truth
    return int(np.sum(a&~b)),int(np.sum(b&~a))

In [8]:
assert paired_disagreements([0,1],[0,0],[1,1])==(1,1)
frame=pd.read_csv(ROOT/'reports/stage_12/test_predictions.csv')
a_only,b_only=paired_disagreements(frame.true_label,frame.pixels,frame.combined)
assert b_only-a_only==int((frame.combined==frame.true_label).sum()-(frame.pixels==frame.true_label).sum())
print("Previously exposed test: pixels-only correct / combined-only correct:",a_only,b_only)

Previously exposed test: pixels-only correct / combined-only correct: 3 7


**Why this works and what it does not establish:** A paired comparison retains information that comparing two accuracy percentages loses. A significance test would additionally require its own assumptions and analysis protocol; this count alone is not one.

## 12.C4 — Fingerprint the evidence

Return the SHA-256 hexadecimal digest of a local file without modifying it.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Hash raw bytes, not a reformatted view of the data.

</details>

In [9]:
def file_fingerprint(path):
    from hashlib import sha256
    return sha256(Path(path).read_bytes()).hexdigest()

In [10]:
p=ROOT/'data/raw/iris.csv'
h=file_fingerprint(p)
assert len(h)==64 and h==file_fingerprint(p)
print("Iris snapshot SHA-256:",h)

Iris snapshot SHA-256: 160191b2e9c845b2478a7e6722acdb8ad6eb54c82a5353972fb71b931772fe4f


**Why this works and what it does not establish:** A content hash detects a change to the file but cannot prove that the original measurement was correct or that the data source was trustworthy.

## 12.C5 — Check a report has its required fields

Return the set of missing field names among source, observation_unit, metric, filtration, coefficients, evaluation_unit, limitations. Do not treat field presence as factual verification.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Compare a set of required keys against the actual keys.

</details>

In [11]:
def missing_report_fields(report):
    required={'source','observation_unit','metric','filtration','coefficients','evaluation_unit','limitations'}
    return required-set(report)

In [12]:
assert 'metric' in missing_report_fields({'source':'example'})
report=dict.fromkeys(['source','observation_unit','metric','filtration','coefficients','evaluation_unit','limitations'],'declared')
assert missing_report_fields(report)==set()
print("Report structure checked; claim correctness still needs review")

Report structure checked; claim correctness still needs review


**Why this works and what it does not establish:** Completeness of a form is not completeness of an argument. Review whether each statement is true, supported, internally consistent, and adequate for the question.